# 04 - Clasificacion: Propina Generosa

## Pregunta de negocio: Podemos clasificar si un pasajero dejara propina generosa (>20%)?

En este notebook construiremos modelos de clasificacion binaria para predecir si un pasajero
dejara una propina generosa (mayor al 20% del total de la tarifa). Este tipo de modelo podria
ayudar a los conductores a optimizar su servicio o a las plataformas a personalizar la experiencia.

### Modelos a explorar:
- Regresion Logistica (baseline)
- Random Forest Classifier
- XGBoost Classifier

### Tecnicas avanzadas:
- Curvas ROC y Precision-Recall
- Ajuste de umbral de decision
- SMOTE para balanceo de clases

**Dataset:** `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`

## 1. Carga de datos y creacion del target binario

In [ ]:
# Imports principales
import sys
sys.path.insert(0, '../../../src')
from bigquery.bq_helper import BigQueryHelper
bq = BigQueryHelper()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuracion de visualizacion
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('Imports cargados correctamente')

In [ ]:
# Consulta: solo pagos con tarjeta de credito (payment_type=1)
# ya que las propinas en efectivo no se registran
query = """
SELECT
    EXTRACT(HOUR FROM pickup_datetime) AS pickup_hour,
    EXTRACT(DAYOFWEEK FROM pickup_datetime) AS pickup_dow,
    EXTRACT(MONTH FROM pickup_datetime) AS pickup_month,
    trip_distance,
    passenger_count,
    fare_amount,
    tolls_amount,
    tip_amount,
    total_amount,
    CASE WHEN rate_code IN (2, 3) THEN 1 ELSE 0 END AS is_airport,
    TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, MINUTE) AS trip_duration_min,
    CASE
        WHEN trip_distance > 0 AND TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, MINUTE) > 0
        THEN trip_distance / (TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, MINUTE) / 60.0)
        ELSE 0
    END AS avg_speed_mph
FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
WHERE
    payment_type = '1'
    AND fare_amount > 2.5
    AND fare_amount < 200
    AND trip_distance > 0
    AND trip_distance < 50
    AND tip_amount >= 0
    AND total_amount > 0
    AND passenger_count > 0
    AND TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, MINUTE) BETWEEN 1 AND 180
ORDER BY RAND()
LIMIT 200000
"""

df = bq.query_to_dataframe(query)
print(f'Registros cargados: {len(df):,}')
df.head()

In [ ]:
# Crear variable target: propina generosa = tip_pct > 20%
# Solo consideramos pagos con tarjeta porque las propinas en efectivo no se registran
df['tip_pct'] = (df['tip_amount'] / df['fare_amount']) * 100

# Filtrar valores extremos de tip_pct
df = df[(df['tip_pct'] >= 0) & (df['tip_pct'] <= 100)].copy()

# Variable target binaria
df['high_tip'] = (df['tip_pct'] > 20).astype(int)

print(f'Registros finales: {len(df):,}')
print(f'\nDistribucion de tip_pct:')
print(df['tip_pct'].describe())
print(f'\nMediana de tip_pct: {df["tip_pct"].median():.1f}%')

## 2. Balance de clases

Antes de entrenar cualquier modelo, es fundamental entender la distribucion de nuestra variable
target. Un desbalance significativo puede sesgar los modelos hacia la clase mayoritaria.

**Conceptos clave:**
- **Clases balanceadas:** cada clase tiene aproximadamente la misma cantidad de muestras
- **Clases desbalanceadas:** una clase domina significativamente (ej. 95% vs 5%)
- Con ~30-40% de propinas generosas, tenemos un desbalance moderado

In [ ]:
# Distribucion de la variable target
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Conteo de clases
class_counts = df['high_tip'].value_counts()
class_pcts = df['high_tip'].value_counts(normalize=True) * 100

colors = ['#3498db', '#e74c3c']
labels = ['No generosa (<=20%)', 'Generosa (>20%)']

axes[0].bar(labels, class_counts.values, color=colors, edgecolor='black')
for i, (count, pct) in enumerate(zip(class_counts.values, class_pcts.values)):
    axes[0].text(i, count + 500, f'{count:,}\n({pct:.1f}%)', ha='center', fontweight='bold')
axes[0].set_title('Distribucion de Clases: Propina Generosa', fontweight='bold')
axes[0].set_ylabel('Cantidad de viajes')

# Distribucion del porcentaje de propina
axes[1].hist(df['tip_pct'], bins=50, color='#2ecc71', edgecolor='black', alpha=0.7)
axes[1].axvline(x=20, color='red', linestyle='--', linewidth=2, label='Umbral 20%')
axes[1].set_title('Distribucion del Porcentaje de Propina', fontweight='bold')
axes[1].set_xlabel('Propina (%)')
axes[1].set_ylabel('Frecuencia')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Clase 0 (No generosa): {class_counts[0]:,} ({class_pcts[0]:.1f}%)')
print(f'Clase 1 (Generosa):    {class_counts[1]:,} ({class_pcts[1]:.1f}%)')
print(f'\nRatio de desbalance: {class_counts[0]/class_counts[1]:.2f}:1')

## 3. Preparacion de datos: Train/Test split con estratificacion

Usamos `stratify` para mantener la misma proporcion de clases en train y test.
Esto es especialmente importante cuando hay desbalance de clases.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Features para clasificacion
feature_cols = [
    'pickup_hour', 'pickup_dow', 'pickup_month',
    'trip_distance', 'passenger_count', 'fare_amount',
    'tolls_amount', 'is_airport', 'trip_duration_min', 'avg_speed_mph'
]

X = df[feature_cols].copy()
y = df['high_tip'].copy()

# Manejar valores faltantes
X = X.fillna(X.median())

# Split estratificado: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {len(X_train):,} muestras')
print(f'Test:  {len(X_test):,} muestras')
print(f'\nProporcion clase positiva en train: {y_train.mean():.3f}')
print(f'Proporcion clase positiva en test:  {y_test.mean():.3f}')

# Escalar features para Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 4. Modelo 1: Regresion Logistica (Baseline)

La regresion logistica es el modelo de referencia para clasificacion binaria.
Es rapido, interpretable y nos da una linea base contra la cual comparar modelos mas complejos.

**Ventajas:** Interpretable, rapido, funciona bien con features linealmente separables.

**Limitaciones:** Asume relacion lineal entre features y log-odds del target.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Entrenar Logistic Regression
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced'  # Compensar desbalance
)
lr_model.fit(X_train_scaled, y_train)

# Predicciones
y_pred_lr = lr_model.predict(X_test_scaled)
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

print('=== Regresion Logistica ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}')
print()
print(classification_report(y_test, y_pred_lr, target_names=['No generosa', 'Generosa']))

In [ ]:
# Coeficientes del modelo logistico: interpretacion
coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coeficiente': lr_model.coef_[0],
    'Odds_Ratio': np.exp(lr_model.coef_[0])
}).sort_values('Coeficiente', key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c' if c < 0 else '#2ecc71' for c in coef_df['Coeficiente']]
ax.barh(coef_df['Feature'], coef_df['Coeficiente'], color=colors, edgecolor='black')
ax.set_title('Coeficientes de Regresion Logistica', fontweight='bold')
ax.set_xlabel('Coeficiente (estandarizado)')
ax.axvline(x=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

print('\nInterpretacion de Odds Ratios:')
print('Un OR > 1 indica que la feature aumenta la probabilidad de propina generosa')
print(coef_df[['Feature', 'Odds_Ratio']].to_string(index=False))

## 5. Modelo 2: Random Forest Classifier con ajuste de hiperparametros

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# Random Forest con busqueda de hiperparametros
rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [8, 12, 16],
    'min_samples_split': [10, 20],
    'min_samples_leaf': [5, 10],
    'class_weight': ['balanced']
}

rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)

rf_grid = GridSearchCV(
    rf_base, rf_params,
    cv=3, scoring='f1',
    n_jobs=-1, verbose=1
)
rf_grid.fit(X_train, y_train)  # RF no necesita escalado

rf_model = rf_grid.best_estimator_
print(f'\nMejores hiperparametros: {rf_grid.best_params_}')
print(f'Mejor F1 (CV): {rf_grid.best_score_:.4f}')

In [ ]:
# Evaluacion Random Forest
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

print('=== Random Forest Classifier ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}')
print()
print(classification_report(y_test, y_pred_rf, target_names=['No generosa', 'Generosa']))

## 6. Modelo 3: XGBoost Classifier con GridSearchCV

XGBoost es uno de los algoritmos mas poderosos para datos tabulares.
Usa gradient boosting, construyendo arboles secuenciales que corrigen los errores
de los arboles anteriores.

In [ ]:
from xgboost import XGBClassifier

# Calcular scale_pos_weight para desbalance
scale_pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

xgb_params = {
    'n_estimators': [100, 200],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8],
    'colsample_bytree': [0.8],
}

xgb_base = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)

xgb_grid = GridSearchCV(
    xgb_base, xgb_params,
    cv=3, scoring='f1',
    n_jobs=-1, verbose=1
)
xgb_grid.fit(X_train, y_train)

xgb_model = xgb_grid.best_estimator_
print(f'\nMejores hiperparametros: {xgb_grid.best_params_}')
print(f'Mejor F1 (CV): {xgb_grid.best_score_:.4f}')

In [ ]:
# Evaluacion XGBoost
y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

print('=== XGBoost Classifier ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}')
print()
print(classification_report(y_test, y_pred_xgb, target_names=['No generosa', 'Generosa']))

## 7. Curvas ROC: Comparacion de modelos

La curva ROC (Receiver Operating Characteristic) muestra la relacion entre
la tasa de verdaderos positivos (TPR/Recall) y la tasa de falsos positivos (FPR)
para diferentes umbrales de decision.

**AUC** (Area Under the Curve): Un modelo perfecto tiene AUC = 1.0, uno aleatorio tiene AUC = 0.5.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

fig, ax = plt.subplots(figsize=(10, 8))

models_probs = {
    'Regresion Logistica': y_prob_lr,
    'Random Forest': y_prob_rf,
    'XGBoost': y_prob_xgb
}
colors_roc = ['#3498db', '#2ecc71', '#e74c3c']

for (name, y_prob), color in zip(models_probs.items(), colors_roc):
    fpr, tpr, thresholds = roc_curve(y_test, y_prob)
    auc_score = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC = {auc_score:.4f})')

# Linea de referencia (clasificador aleatorio)
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Aleatorio (AUC = 0.5)')

ax.set_xlabel('Tasa de Falsos Positivos (FPR)', fontsize=13)
ax.set_ylabel('Tasa de Verdaderos Positivos (TPR)', fontsize=13)
ax.set_title('Curvas ROC - Comparacion de Modelos', fontweight='bold', fontsize=14)
ax.legend(loc='lower right', fontsize=11)
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
plt.tight_layout()
plt.show()

## 8. Curvas Precision-Recall

Las curvas PR son mas informativas que las ROC cuando las clases estan desbalanceadas.
Muestran el trade-off entre precision (cuantos de los predichos positivos son correctos)
y recall (cuantos de los positivos reales detectamos).

**Average Precision (AP):** resumen de la curva PR, equivalente al area bajo la curva PR.

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

fig, ax = plt.subplots(figsize=(10, 8))

for (name, y_prob), color in zip(models_probs.items(), colors_roc):
    precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    ax.plot(recall, precision, color=color, linewidth=2, label=f'{name} (AP = {ap:.4f})')

# Linea base: proporcion de positivos
baseline = y_test.mean()
ax.axhline(y=baseline, color='gray', linestyle='--', linewidth=1,
           label=f'Aleatorio (AP = {baseline:.3f})')

ax.set_xlabel('Recall', fontsize=13)
ax.set_ylabel('Precision', fontsize=13)
ax.set_title('Curvas Precision-Recall - Comparacion de Modelos', fontweight='bold', fontsize=14)
ax.legend(loc='best', fontsize=11)
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([0, 1.05])
plt.tight_layout()
plt.show()

print('La curva PR es especialmente util cuando:')
print('- Las clases estan desbalanceadas')
print('- Nos importa mas la precision en la clase positiva')
print('- Los falsos positivos tienen un costo alto')

## 9. Ajuste de umbral de decision

Por defecto, los clasificadores usan un umbral de 0.5 para decidir la clase.
Pero el umbral optimo depende del **objetivo de negocio**:

- Si queremos **maximizar recall** (detectar todas las propinas generosas): bajar el umbral
- Si queremos **maximizar precision** (estar seguros cuando predecimos generosa): subir el umbral
- Para un balance: buscar el umbral que maximiza F1

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

# Usamos el mejor modelo (XGBoost) para ajustar el umbral
thresholds_range = np.arange(0.1, 0.9, 0.01)

metrics_by_threshold = []
for t in thresholds_range:
    y_pred_t = (y_prob_xgb >= t).astype(int)
    metrics_by_threshold.append({
        'threshold': t,
        'precision': precision_score(y_test, y_pred_t, zero_division=0),
        'recall': recall_score(y_test, y_pred_t, zero_division=0),
        'f1': f1_score(y_test, y_pred_t, zero_division=0),
        'accuracy': accuracy_score(y_test, y_pred_t)
    })

metrics_df = pd.DataFrame(metrics_by_threshold)

# Visualizar
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(metrics_df['threshold'], metrics_df['precision'], label='Precision', linewidth=2)
ax.plot(metrics_df['threshold'], metrics_df['recall'], label='Recall', linewidth=2)
ax.plot(metrics_df['threshold'], metrics_df['f1'], label='F1-Score', linewidth=2, linestyle='--')
ax.plot(metrics_df['threshold'], metrics_df['accuracy'], label='Accuracy', linewidth=2, linestyle=':')

# Umbral optimo (max F1)
best_idx = metrics_df['f1'].idxmax()
best_threshold = metrics_df.loc[best_idx, 'threshold']
best_f1 = metrics_df.loc[best_idx, 'f1']
ax.axvline(x=best_threshold, color='red', linestyle='--', alpha=0.7,
           label=f'Umbral optimo = {best_threshold:.2f} (F1={best_f1:.3f})')

ax.set_xlabel('Umbral de Decision', fontsize=13)
ax.set_ylabel('Score', fontsize=13)
ax.set_title('Metricas vs. Umbral de Decision (XGBoost)', fontweight='bold', fontsize=14)
ax.legend(loc='best', fontsize=11)
plt.tight_layout()
plt.show()

print(f'Umbral optimo (max F1): {best_threshold:.2f}')
print(f'  Precision: {metrics_df.loc[best_idx, "precision"]:.3f}')
print(f'  Recall:    {metrics_df.loc[best_idx, "recall"]:.3f}')
print(f'  F1:        {best_f1:.3f}')

## 10. SMOTE: Balanceo de clases con oversampling sintetico

**SMOTE** (Synthetic Minority Over-sampling Technique) genera muestras sinteticas
de la clase minoritaria interpolando entre ejemplos existentes cercanos.

**Importante:** SMOTE solo se aplica al conjunto de **entrenamiento**, nunca al test.
El test debe reflejar la distribucion real de los datos.

In [ ]:
from imblearn.over_sampling import SMOTE

# Aplicar SMOTE solo al train
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print('Antes de SMOTE:')
print(f'  Clase 0: {(y_train == 0).sum():,}')
print(f'  Clase 1: {(y_train == 1).sum():,}')
print(f'\nDespues de SMOTE:')
print(f'  Clase 0: {(y_train_smote == 0).sum():,}')
print(f'  Clase 1: {(y_train_smote == 1).sum():,}')
print(f'\nMuestras sinteticas generadas: {len(y_train_smote) - len(y_train):,}')

In [ ]:
# Reentrenar XGBoost con datos balanceados por SMOTE
xgb_smote = XGBClassifier(
    **{k: v for k, v in xgb_grid.best_params_.items()},
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)
xgb_smote.fit(X_train_smote, y_train_smote)

y_pred_smote = xgb_smote.predict(X_test)
y_prob_smote = xgb_smote.predict_proba(X_test)[:, 1]

print('=== XGBoost con SMOTE ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_smote):.4f}')
print()
print(classification_report(y_test, y_pred_smote, target_names=['No generosa', 'Generosa']))

## 11. Matrices de confusion: antes y despues de SMOTE

In [ ]:
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

class_labels = ['No generosa', 'Generosa']

# Antes de SMOTE (XGBoost original)
cm_before = confusion_matrix(y_test, y_pred_xgb)
sns.heatmap(cm_before.astype(float), annot=True, fmt='.0f', cmap='Blues',            xticklabels=class_labels, yticklabels=class_labels, ax=axes[0])
axes[0].set_title('XGBoost SIN SMOTE', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Prediccion')
axes[0].set_ylabel('Real')

# Despues de SMOTE
cm_after = confusion_matrix(y_test, y_pred_smote)
sns.heatmap(cm_after.astype(float), annot=True, fmt='.0f', cmap='Oranges',            xticklabels=class_labels, yticklabels=class_labels, ax=axes[1])
axes[1].set_title('XGBoost CON SMOTE', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Prediccion')
axes[1].set_ylabel('Real')

plt.tight_layout()
plt.show()

print('Comparacion de matrices de confusion:')
print(f'\nSin SMOTE - FP: {cm_before[0,1]:,}, FN: {cm_before[1,0]:,}')
print(f'Con SMOTE - FP: {cm_after[0,1]:,}, FN: {cm_after[1,0]:,}')
print(f'\nSMOTE tipicamente reduce los falsos negativos (mejora recall)')
print(f'pero puede aumentar los falsos positivos (reduce precision)')

## 12. Tabla comparativa de modelos

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

# Recopilar metricas de todos los modelos
results = []
all_models = {
    'Logistic Regression': (y_pred_lr, y_prob_lr),
    'Random Forest': (y_pred_rf, y_prob_rf),
    'XGBoost': (y_pred_xgb, y_prob_xgb),
    'XGBoost + SMOTE': (y_pred_smote, y_prob_smote)
}

for name, (y_pred, y_prob) in all_models.items():
    results.append({
        'Modelo': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'AUC-ROC': roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results)
results_df = results_df.set_index('Modelo')

# Formatear como porcentajes
print('='*70)
print('TABLA COMPARATIVA DE MODELOS - CLASIFICACION PROPINA GENEROSA')
print('='*70)
print(results_df.round(4).to_string())
print('='*70)

# Visualizar
fig, ax = plt.subplots(figsize=(12, 6))
results_df[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']].plot(
    kind='bar', ax=ax, edgecolor='black'
)
ax.set_title('Comparacion de Modelos de Clasificacion', fontweight='bold', fontsize=14)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.1)
ax.legend(loc='lower right')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Identificar mejor modelo
best_model_name = results_df['F1-Score'].idxmax()
print(f'\nMejor modelo por F1-Score: {best_model_name}')

## 13. Guardar el mejor modelo

In [ ]:
import joblib
import os

# Crear directorio si no existe
model_dir = '../../../models/nyc_taxi/'
os.makedirs(model_dir, exist_ok=True)

# Seleccionar el mejor modelo basado en F1
best_models_map = {
    'Logistic Regression': lr_model,
    'Random Forest': rf_model,
    'XGBoost': xgb_model,
    'XGBoost + SMOTE': xgb_smote
}

best_clf = best_models_map[best_model_name]
model_path = os.path.join(model_dir, 'high_tip_classifier.joblib')

# Guardar modelo, scaler y metadata
model_artifact = {
    'model': best_clf,
    'scaler': scaler,
    'feature_cols': feature_cols,
    'model_name': best_model_name,
    'metrics': results_df.loc[best_model_name].to_dict(),
    'optimal_threshold': best_threshold
}

joblib.dump(model_artifact, model_path)
print(f'Modelo guardado en: {model_path}')
print(f'Modelo seleccionado: {best_model_name}')
print(f'Tamano del archivo: {os.path.getsize(model_path) / 1024:.1f} KB')
print(f'\nMetricas del modelo guardado:')
for metric, value in results_df.loc[best_model_name].items():
    print(f'  {metric}: {value:.4f}')

## Conclusiones

### Hallazgos principales:

1. **Balance de clases:** La proporcion de propinas generosas (>20%) es aproximadamente del 30-40%,
   lo que representa un desbalance moderado.

2. **Modelos de ensamble superan al baseline:** Random Forest y XGBoost superan consistentemente
   a la Regresion Logistica, capturando relaciones no lineales.

3. **SMOTE mejora el recall** a costa de la precision. Es util cuando queremos detectar
   la mayor cantidad posible de propinas generosas.

4. **El umbral optimo no es 0.5:** Ajustar el umbral de decision permite optimizar
   el modelo para el objetivo de negocio especifico.

5. **Curvas PR vs ROC:** En datasets con desbalance, las curvas Precision-Recall son
   mas informativas que las curvas ROC.

### Proximos pasos:
- Explorar clasificacion multiclase (notebook 05)
- Interpretar por que el modelo toma sus decisiones (notebook 06)